# Experiment 02 — M500 Baseline Skin-Tone Bias Analysis (ITA)

**Goal:** Quantify the baseline skin-tone distribution of M500.pkl before any fine-tuning.
1. Setup NVlabs/stylegan3 & pull M500.pkl via DVC (using Colab Secrets — no hardcoded tokens).
2. Generate 500 synthetic melanoma images (seeds 0..499, 	runcation_psi=0.7).
3. Measure non-lesion skin tone using src/evaluation/ita.py (Individual Typology Angle).
4. Save & visualize the baseline bias report (
eports/baseline/).


In [ ]:
# 1. Verify GPU runtime
import os, sys, subprocess
from pathlib import Path
import torch

assert torch.cuda.is_available(), 'No GPU detected! Switch Colab Runtime to GPU.'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))


In [ ]:
# 2. Clone StyleGAN3 (for legacy StyleGAN2-ADA loader) & project repo
SG3_DIR = Path('/content/stylegan3')
if not SG3_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/NVlabs/stylegan3.git', str(SG3_DIR)], check=True)
sys.path.insert(0, str(SG3_DIR))

REPO_ROOT = Path('/content/bias-aware-stylegan')
if not REPO_ROOT.exists():
    subprocess.run(['git', 'clone', '-b', 'experiment/m500-baseline', 'https://github.com/Muneeb-Ahmad-404/bias-aware-stylegan.git', str(REPO_ROOT)], check=True)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('Working dir:', Path.cwd())


In [ ]:
# 3. Install lightweight dependencies & pull M500.pkl via DVC (Colab Secrets)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'dvc', 'ninja', 'pyspng', 'scipy', 'opencv-python', 'pandas', 'matplotlib'], check=True)

from google.colab import userdata
dagshub_user = userdata.get('DAGSHUB_USERNAME')
dagshub_token = userdata.get('DAGSHUB_TOKEN')
assert dagshub_user and dagshub_token, 'Set DAGSHUB_USERNAME and DAGSHUB_TOKEN in Colab Secrets!'

subprocess.run([sys.executable, '-m', 'dvc', 'remote', 'modify', '--local', 'origin_processed', 'auth', 'basic'], check=True)
subprocess.run([sys.executable, '-m', 'dvc', 'remote', 'modify', '--local', 'origin_processed', 'user', dagshub_user], check=True)
subprocess.run([sys.executable, '-m', 'dvc', 'remote', 'modify', '--local', 'origin_processed', 'password', dagshub_token], check=True)
subprocess.run([sys.executable, '-m', 'dvc', 'pull', 'models/pretrained/M500.pkl.dvc'], check=True)


In [ ]:
# 4. Load M500.pkl (G_ema) and patch ops to PyTorch ref if nvcc build is skipped
import legacy
import torch_utils.ops.bias_act as bias_act
import torch_utils.ops.upfirdn2d as upfirdn2d

_orig_bias_act = bias_act.bias_act
bias_act.bias_act = lambda *a, **kw: _orig_bias_act(*a, **{**kw, 'impl': 'ref'})
_orig_upfirdn2d = upfirdn2d.upfirdn2d
upfirdn2d.upfirdn2d = lambda *a, **kw: _orig_upfirdn2d(*a, **{**kw, 'impl': 'ref'})

device = torch.device('cuda')
with open('models/pretrained/M500.pkl', 'rb') as f:
    G_ema = legacy.load_network_pkl(f)['G_ema'].to(device).eval()
for p in G_ema.parameters():
    p.requires_grad_(False)
print(f'Loaded G_ema: res={G_ema.img_resolution}, z_dim={G_ema.z_dim}, c_dim={G_ema.c_dim}')


In [ ]:
# 5. Generate 500 baseline samples in batches
import numpy as np
from PIL import Image
from tqdm import tqdm

NUM_SAMPLES = 500
BATCH_SIZE = 10
TRUNCATION_PSI = 0.7
OUT_DIR = REPO_ROOT / 'outputs' / 'm500_baseline_500'
OUT_DIR.mkdir(parents=True, exist_ok=True)

c_zero = torch.zeros([BATCH_SIZE, G_ema.c_dim], device=device)

for start in tqdm(range(0, NUM_SAMPLES, BATCH_SIZE), desc='Generating'):
    seeds = list(range(start, min(start + BATCH_SIZE, NUM_SAMPLES)))
    z_np = np.stack([np.random.RandomState(s).randn(G_ema.z_dim) for s in seeds])
    z = torch.from_numpy(z_np).to(device=device, dtype=torch.float32)
    c = c_zero[:len(seeds)]
    with torch.no_grad():
        imgs = G_ema(z, c, truncation_psi=TRUNCATION_PSI, noise_mode='const')
    imgs_u8 = (imgs.permute(0, 2, 3, 1) * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu().numpy()
    for s, arr in zip(seeds, imgs_u8):
        Image.fromarray(arr).save(OUT_DIR / f'seed_{s:04d}.png')

print('Generated images:', len(list(OUT_DIR.glob('*.png'))))


In [ ]:
# 6. Run ITA skin-tone bias analysis & save report
import json
import matplotlib.pyplot as plt
from src.evaluation.ita import analyze_image_folder, save_ita_report, ITA_BINS

df, summary = analyze_image_folder(OUT_DIR)
report_dir = REPO_ROOT / 'reports' / 'baseline'
paths = save_ita_report(df, summary, report_dir, prefix='m500_ita')

print(json.dumps(summary, indent=2))

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
groups = [b[0] for b in ITA_BINS]
pcts = [summary['distribution'][g]['percentage'] for g in groups]

axes[0].bar(groups, pcts, color=['#fce5cd', '#f9cb9c', '#e6b8af', '#c27ba0', '#7f6000', '#4c1130'])
axes[0].set_title('M500 Baseline Skin-Tone Distribution (N=500)')
axes[0].set_ylabel('Percentage (%)')
axes[0].tick_params(axis='x', rotation=25)

axes[1].hist(df['ita_deg'].dropna(), bins=30, edgecolor='black', alpha=0.8)
axes[1].axvline(28, color='orange', linestyle='--', label='Tan/FST IV threshold (28°)')
axes[1].axvline(10, color='red', linestyle='--', label='Brown/FST V threshold (10°)')
axes[1].set_title('M500 ITA Angle Distribution (degrees)')
axes[1].set_xlabel('ITA (degrees)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
fig.savefig(report_dir / 'm500_ita_distribution.png', dpi=150)
plt.show()
